# Vectorized Backpropagation — NumPy

This is the second notebook in a two-part exercise. The first, 
`backprop-from-scratch-numpy.ipynb`, implements forward propagation and 
backpropagation using explicit nested loops over each neuron and weight — 
written that way deliberately, to make every individual computation visible.

This notebook takes the same 3-layer network and the same math, and replaces 
the loops with matrix operations (`np.matmul`). The goal isn't a different 
network — it's the same network, expressed more efficiently, with a working 
training loop added on top.

**What's the same:** architecture (2→3→2→1), activation (sigmoid throughout), 
loss function (squared error), single training example.

**What's new:** vectorized forward + backward pass, a training loop over 
multiple epochs.

In [53]:
import numpy as np

In [54]:
def sigmoid(z): 
    return 1 / (1 + np.exp(-z))

In [55]:
def my_dense(a_in, W, b, g):
    """
    Args: 
     a_in(ndarray (n,)): 1 example
     W(ndarray (m,n)): Weights for each example
     b(ndarray (n)): bias vector, j units
     g: activation function (sigmoid, relu, etc.)

    Returns:
     a_out(ndarray (n,)): n units
    """
    z = np.matmul(W.T, a_in) + b
    a_out = g(z)
    return z, a_out

In [56]:
x_tst = 0.1*np.arange(1,3,1).reshape(2,)  # (1 examples, 2 features)
W_tst = 0.1*np.arange(1,7,1).reshape(2,3) # (2 input features, 3 output features)
b_tst = 0.1*np.arange(1,4,1).reshape(3,)  # (3 features)
A_tst = my_dense(x_tst, W_tst, b_tst, sigmoid)
print(A_tst)

(array([0.19, 0.32, 0.45]), array([0.54735762, 0.57932425, 0.61063923]))


In [57]:
def my_sequential(X, W1, b1, W2, b2, W3, b3):
    z1, a1 = my_dense(X, W1, b1, sigmoid)
    z2, a2 = my_dense(a1, W2, b2, sigmoid)
    z3, a3 = my_dense(a2, W3, b3, sigmoid)
    return z1, a1, z2, a2, z3, a3

In [58]:
X = np.array([1.0, 2.0])

# Layer 1: 2 inputs to 3 neurons
W1 = np.array([
    [0.1, 0.2, 0.3],
    [0.4, 0.5, 0.6]
])
b1 = np.array([0.1, 0.1, 0.1])

# Layer 2: 3 neurons to 2 neurons
W2 = np.array([
    [0.1, 0.2],
    [0.3, 0.4],
    [0.5, 0.6]
])
b2 = np.array([0.1, 0.1])

# Layer 3: 2 neurons to 1 neuron
W3 = np.array([
    [0.7],
    [0.8]
])
b3 = np.array([0.1])

z1, a1, z2, a2, z3, a3 = my_sequential(X, W1, b1, W2, b2, W3, b3)
print(a3)

[0.76509215]


In [59]:
y = np.array([1.0]) # Suppose y = 1.0

In [60]:
def backprop(X, y, z1, a1, z2, a2, z3, a3, W1, W2, W3): 
    dJ_da3 = (a3 - y)
    dJ_dz3 = dJ_da3 * a3 * (1 - a3) 
    
    dJ_dW3 = np.matmul(
        a2.reshape(-1, 1),
        dJ_dz3.reshape(1, -1)
    )
    
    dJ_db3 = dJ_dz3
    
    dJ_da2 = np.matmul(W3, dJ_dz3)
    dJ_dz2 = dJ_da2 * a2 * (1 - a2)
    
    dJ_dW2 = np.matmul(
        a1.reshape(-1, 1),
        dJ_dz2.reshape(1, -1)
    )
    
    dJ_db2 = dJ_dz2 
    
    dJ_da1 = np.matmul(W2, dJ_dz2)
    dJ_dz1 = dJ_da1 * a1 * (1 - a1)
    
    dJ_dW1 = np.matmul(
        X.reshape(-1, 1),
        dJ_dz1.reshape(1, -1)
    )
    
    dJ_db1 = dJ_dz1

    return (
        dJ_dW1, dJ_db1,
        dJ_dW2, dJ_db2,
        dJ_dW3, dJ_db3
    )

In [61]:
alpha = 0.1
epochs = 100

In [62]:
for epoch in range(epochs):
    z1,a1,z2,a2,z3,a3 = my_sequential(
        X,
        W1,b1,
        W2,b2,
        W3,b3
    )
    
    J = 0.5 * np.sum((a3-y)**2)

    (
        dJ_dW1, dJ_db1,
        dJ_dW2, dJ_db2,
        dJ_dW3, dJ_db3
    ) = backprop(
        X, y,
        z1,a1,
        z2,a2,
        z3,a3,
        W1,W2,W3
    )

    W1 -= alpha*dJ_dW1
    b1 -= alpha*dJ_db1

    W2 -= alpha*dJ_dW2
    b2 -= alpha*dJ_db2

    W3 -= alpha*dJ_dW3
    b3 -= alpha*dJ_db3

    if epoch % 2 == 0:
        print(f"Epoch: {epoch}, Loss: {J}")

Epoch: 0, Loss: 0.027590849084389118
Epoch: 2, Loss: 0.026830496066576084
Epoch: 4, Loss: 0.02610384061783425
Epoch: 6, Loss: 0.02540894654701275
Epoch: 8, Loss: 0.024744008398604043
Epoch: 10, Loss: 0.024107341759634724
Epoch: 12, Loss: 0.02349737430200198
Epoch: 14, Loss: 0.022912637509942085
Epoch: 16, Loss: 0.02235175904406019
Epoch: 18, Loss: 0.021813455695486213
Epoch: 20, Loss: 0.0212965268861073
Epoch: 22, Loss: 0.02079984867335193
Epoch: 24, Loss: 0.020322368220580592
Epoch: 26, Loss: 0.019863098696713013
Epoch: 28, Loss: 0.019421114571243615
Epoch: 30, Loss: 0.018995547273236133
Epoch: 32, Loss: 0.018585581185221554
Epoch: 34, Loss: 0.018190449945138524
Epoch: 36, Loss: 0.017809433031541234
Epoch: 38, Loss: 0.01744185260925691
Epoch: 40, Loss: 0.017087070614500655
Epoch: 42, Loss: 0.016744486060152174
Epoch: 44, Loss: 0.01641353254347266
Epoch: 46, Loss: 0.016093675939994636
Epoch: 48, Loss: 0.01578441226865895
Epoch: 50, Loss: 0.015485265714508901
Epoch: 52, Loss: 0.01519578